# Public-release note

This curated notebook contains the original research workflow with execution outputs removed. Bloomberg and other licensed source data are not distributed. To run it, supply compatible files under `data/processed/` as described in `data/README.md`. Any results produced locally depend on the user's licensed data and are not included in this repository.


# Sharpe-Ratio Significance Tests

This notebook applies the Jobson-Korkie test with Memmel correction as a parametric robustness check for differences in Sharpe ratios. The stationary bootstrap remains the primary inference framework.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Locate the repository root when running from either the project or notebooks directory
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.metrics.performance_metrics import jobson_korkie_memmel_test
from src.portfolio.rebalancing import run_rebalancing_strategy

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "jobson_korkie"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
COUNTRIES = {"US": "us_data", "UK": "uk_data", "DE": "de_data"}
HORIZONS = [5, 10]

SAMPLES_60_40 = {
    "paper_sample": ("1982-01-31", "2011-12-31"),
    "post_2011": ("2012-01-31", "2026-05-31"),
}

SAMPLES_MULTI_ASSET = {
    "paper_sample": ("1987-03-31", "2011-12-31"),
    "post_2011": ("2012-01-31", "2026-05-31"),
}

# Compare rebalancing strategies with buy-and-hold using the replication design
STRATEGIES = {
    "Yearly periodic": {"strategy": "periodic", "frequency": "Y", "comparison": "Y-PER-BAH"},
    "Quarterly periodic": {"strategy": "periodic", "frequency": "Q", "comparison": "Q-PER-BAH"},
    "Monthly periodic": {"strategy": "periodic", "frequency": "M", "comparison": "M-PER-BAH"},
    "Yearly threshold": {"strategy": "threshold", "frequency": "Y", "comparison": "Y-THR-BAH"},
    "Quarterly threshold": {"strategy": "threshold", "frequency": "Q", "comparison": "Q-THR-BAH"},
    "Monthly threshold": {"strategy": "threshold", "frequency": "M", "comparison": "M-THR-BAH"},
    "Yearly range": {"strategy": "range", "frequency": "Y", "comparison": "Y-RNG-BAH"},
    "Quarterly range": {"strategy": "range", "frequency": "Q", "comparison": "Q-RNG-BAH"},
    "Monthly range": {"strategy": "range", "frequency": "M", "comparison": "M-RNG-BAH"},
}


In [ ]:
def sample_filter(df: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    return df[df["Date"].between(pd.Timestamp(start), pd.Timestamp(end))].copy()


def result_label(diff: float, p_value: float) -> str:
    if pd.isna(p_value) or p_value > 0.10:
        return "not significant"
    return "positive" if diff > 0 else "negative"


def jk_result(ret_1: pd.Series, ret_2: pd.Series, rf: pd.Series, **meta) -> dict:
    stats = jobson_korkie_memmel_test(ret_1, ret_2, rf=rf)
    return meta | stats | {"result": result_label(stats["sharpe_diff"], stats["p_value"])}


## 60/40 replication and extension

In [ ]:
def strategy_returns_60_40(df: pd.DataFrame, horizon: int, start: str, end: str) -> pd.DataFrame:
    bond_col = f"Bond_{horizon}Y_Return"
    data = sample_filter(df, start, end).dropna(subset=["Equity_Return", bond_col, "RF_Return"])
    out = {
        "Buy-and-hold": run_rebalancing_strategy(
            "buy_and_hold", data["Equity_Return"], data[bond_col], data["Date"]
        ).set_index("Date")["Portfolio_Return"]
    }

    for name, cfg in STRATEGIES.items():
        out[name] = run_rebalancing_strategy(
            cfg["strategy"],
            data["Equity_Return"],
            data[bond_col],
            data["Date"],
            frequency=cfg["frequency"],
            threshold=0.03,
        ).set_index("Date")["Portfolio_Return"]

    return pd.DataFrame(out).join(data.set_index("Date")["RF_Return"])


def run_60_40_tests() -> pd.DataFrame:
    rows = []
    for sample, dates in SAMPLES_60_40.items():
        for country, file_name in COUNTRIES.items():
            df = pd.read_csv(DATA_DIR / f"{file_name}.csv", parse_dates=["Date"])
            for horizon in HORIZONS:
                rets = strategy_returns_60_40(df, horizon, *dates)
                for strategy, cfg in STRATEGIES.items():
                    rows.append(
                        jk_result(
                            rets[strategy],
                            rets["Buy-and-hold"],
                            rets["RF_Return"],
                            Section="60_40",
                            Sample=sample,
                            Country=country,
                            Horizon=horizon,
                            Method="60/40",
                            Strategy_1=strategy,
                            Strategy_2="Buy-and-hold",
                            Comparison=cfg["comparison"],
                        )
                    )
    return pd.DataFrame(rows)


jk_60_40 = run_60_40_tests()
jk_60_40.head()


## GMV and Risk Parity

In [ ]:
def run_multi_asset_tests() -> pd.DataFrame:
    monthly = pd.read_csv(RESULTS_DIR.parent / "gmv_risk_parity" / "gmv_risk_parity_rebalancing_monthly.csv", parse_dates=["Date"])
    rf = pd.read_csv(DATA_DIR / "multi_asset_usd_returns.csv", parse_dates=["Date"])[["Date", "Cash_USD_Return"]]
    monthly = monthly.merge(rf, on="Date", how="left")
    rows = []

    for sample, dates in SAMPLES_MULTI_ASSET.items():
        data = sample_filter(monthly, *dates)
        for method in ["GMV", "Risk Parity"]:
            sub = data[data["Method"].eq(method)]
            rets = sub.pivot(index="Date", columns="Strategy", values="Portfolio_Return")
            cash = sub.drop_duplicates("Date").set_index("Date")["Cash_USD_Return"]
            for strategy, cfg in STRATEGIES.items():
                rows.append(
                    jk_result(
                        rets[strategy],
                        rets["Buy-and-hold"],
                        cash,
                        Section="GMV_Risk_Parity",
                        Sample=sample,
                        Country="Global_USD",
                        Horizon="",
                        Method=method,
                        Strategy_1=strategy,
                        Strategy_2="Buy-and-hold",
                        Comparison=cfg["comparison"],
                    )
                )
    return pd.DataFrame(rows)


jk_gmv_rp = run_multi_asset_tests()
jk_gmv_rp.head()


## Summary and export

In [ ]:
jk_all = pd.concat([jk_60_40, jk_gmv_rp], ignore_index=True)
ordered_cols = [
    "Section", "Sample", "Country", "Horizon", "Method", "Comparison",
    "Strategy_1", "Strategy_2", "n_obs", "sharpe_1", "sharpe_2",
    "sharpe_diff", "rho", "z_stat", "p_value", "significance", "result",
]
jk_all = jk_all[ordered_cols].sort_values(["Section", "Sample", "Country", "Horizon", "Method", "Comparison"])

# Compact count of positive, negative, and statistically significant differences
summary = (
    jk_all.assign(Significant_10pct=jk_all["p_value"].le(0.10))
    .groupby(["Section", "Sample", "Method", "result"], as_index=False)
    .agg(N_Tests=("Comparison", "count"), Significant_10pct=("Significant_10pct", "sum"), Avg_Sharpe_Diff=("sharpe_diff", "mean"))
)

jk_all.head(), summary


In [ ]:
path_60_40 = RESULTS_DIR / "jobson_korkie_60_40.csv"
path_gmv_rp = RESULTS_DIR / "jobson_korkie_gmv_risk_parity.csv"
path_all = RESULTS_DIR / "jobson_korkie_all_tests.csv"
path_summary = RESULTS_DIR / "jobson_korkie_summary.csv"
xlsx_path = RESULTS_DIR / "jobson_korkie_sharpe_tests.xlsx"

jk_60_40.to_csv(path_60_40, index=False)
jk_gmv_rp.to_csv(path_gmv_rp, index=False)
jk_all.to_csv(path_all, index=False)
summary.to_csv(path_summary, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    jk_all.to_excel(writer, sheet_name="all_tests", index=False)
    jk_60_40.to_excel(writer, sheet_name="60_40", index=False)
    jk_gmv_rp.to_excel(writer, sheet_name="gmv_risk_parity", index=False)
    summary.to_excel(writer, sheet_name="summary", index=False)

pd.DataFrame({
    "File": [path_60_40.name, path_gmv_rp.name, path_all.name, path_summary.name, xlsx_path.name],
    "Path": [path_60_40, path_gmv_rp, path_all, path_summary, xlsx_path],
})


Interpretation: `sharpe_diff` is positive when a rebalancing strategy has a higher Sharpe ratio than buy-and-hold. Significance markers follow the convention `*` 10%, `**` 5%, and `***` 1%.
